# ניסוי: ניקוד אסתטי + קיבוץ תמונות

**הוראות (3 דקות):**
1. למעלה: `Runtime` → `Change runtime type` → בחר **GPU** → Save.
2. הרץ את התאים לפי הסדר (Shift+Enter על כל תא, או `Runtime` → `Run all`).
3. בתא ההעלאה — בחר 100–200 תמונות מהמחשב (רצוי כאלה שאתה כבר יודע אילו טובות).
4. בסוף תראה גלריה ממוינת מהטוב לפחות, עם מספר קבוצה לכל תמונה.

אם המיון תואם לעין שלך — הרעיון עובד ושווה לבנות. אפס עלות, רץ אצל גוגל.

In [ ]:
#@title 1) התקנת ספריות (חצי דקה)
!pip -q install open_clip_torch scikit-learn pillow >/dev/null
print('done')

In [ ]:
#@title 2) טעינת המודלים (CLIP + ראש אסתטי)
import torch, torch.nn as nn, open_clip

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
model = model.to(device).eval()

class AestheticHead(nn.Module):
    def __init__(self, d=768):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d,1024), nn.Dropout(0.2),
            nn.Linear(1024,128), nn.Dropout(0.2),
            nn.Linear(128,64), nn.Dropout(0.1),
            nn.Linear(64,16), nn.Linear(16,1))
    def forward(self,x): return self.layers(x)

url = 'https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac+logos+ava1-l14-linearMSE.pth'
head = AestheticHead().to(device)
head.load_state_dict(torch.hub.load_state_dict_from_url(url, map_location='cpu'))
head.eval()
print('models ready')

In [ ]:
#@title 3) העלאת התמונות שלך
from google.colab import files
import os
os.makedirs('photos', exist_ok=True)
up = files.upload()
for name, data in up.items():
    open(f'photos/{name}', 'wb').write(data)
print(f'{len(up)} תמונות הועלו')

In [ ]:
#@title 4) ניקוד + קיבוץ
import numpy as np, glob
from PIL import Image
from sklearn.cluster import AgglomerativeClustering

paths = [p for p in glob.glob('photos/*') if p.lower().split('.')[-1] in ('jpg','jpeg','png','webp','bmp','tiff')]
emb, raw, kept = [], [], []
with torch.no_grad():
    for i in range(0, len(paths), 16):
        batch = paths[i:i+16]; tensors, ok = [], []
        for p in batch:
            try: tensors.append(preprocess(Image.open(p).convert('RGB'))); ok.append(p)
            except Exception as e: print('skip', p, e)
        if not tensors: continue
        x = torch.stack(tensors).to(device)
        f = model.encode_image(x); f = f / f.norm(dim=-1, keepdim=True)
        s = head(f.float()).squeeze(-1)
        emb += list(f.cpu().numpy()); raw += s.cpu().numpy().tolist(); kept += ok
        print(f'{len(kept)}/{len(paths)}')

emb = np.array(emb); raw = np.array(raw)
lo, hi = raw.min(), raw.max()
norm = (raw-lo)/(hi-lo) if hi>lo else np.zeros_like(raw)
labels = AgglomerativeClustering(n_clusters=None, distance_threshold=0.18, metric='cosine', linkage='average').fit_predict(emb) if len(kept)>1 else np.zeros(len(kept),int)
print(f'{len(set(labels))} קבוצות')

In [ ]:
#@title 5) הצגת הגלריה הממוינת (הטוב ביותר למעלה)
import base64
from IPython.display import HTML, display
rows = sorted(zip(kept, norm.tolist(), labels.tolist()), key=lambda r: r[1], reverse=True)
cards = []
for p, n, c in rows:
    b64 = base64.b64encode(open(p,'rb').read()).decode()
    cards.append(f'<figure style="margin:0;background:#1c1c1c;border-radius:10px;overflow:hidden">'
                 f'<img src="data:image/jpeg;base64,{b64}" style="width:100%;height:170px;object-fit:cover;display:block">'
                 f'<figcaption style="padding:6px 9px;font-size:13px;color:#eee;display:flex;justify-content:space-between">'
                 f'<b style="color:#7fd">{n*5:.1f}/5</b><span style="color:#9aa;font-size:11px">קבוצה {c}</span></figcaption></figure>')
display(HTML(f'<div dir="rtl" style="display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:12px">{"".join(cards)}</div>'))

import csv
with open('scores.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['file','score_0_1','score_0_5','cluster'])
    for p,n,c in rows: w.writerow([p.split('/')[-1], f'{n:.4f}', f'{n*5:.2f}', c])
print('נשמר scores.csv — אפשר להוריד מהסרגל הצדדי')